# Silver → Gold: Enrich and Export to Parquet

This notebook reads **already cleaned** piece data from the silver layer (`silver.clean_pieces`), enriches it with computed features, and exports to a parquet file ready for analytics and ML.

### What happens here (enrichment only — no cleaning)

All data quality cleaning was done in the bronze → silver step. This notebook only adds analytical value:

1. Compute partial times between process stages (inter-stage durations)
2. Mark production gaps and assign production run IDs
3. Export to `data/gold/pieces.parquet`

### Why this regenerates fully

Production run IDs depend on the ordering of all pieces globally. Adding new data changes the run boundaries, so the gold output is always rebuilt from the complete silver table.

In [1]:
# TODO: implement this cell
import os
import pandas as pd
from sqlalchemy import create_engine

# Load env vars from infra/.env
for line in open("../infra/.env"):
    line = line.strip()
    if line and not line.startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k] = v

engine = create_engine(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)


## Load silver data

Read the full `silver.clean_pieces` table. All cleaning (zeros, upsetting, outliers, monotonic order) was already applied.

In [2]:
# TODO: implement this cell
silver = pd.read_sql("""
SELECT *
FROM silver.clean_pieces
ORDER BY timestamp
""", engine)

silver.head()


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_bath_s,lifetime_general_s,processed_at,oee_cycle_time_s,lifetime_auxiliary_press_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,56.200001,56.200001,2026-04-12 10:57:31.136810+00:00,NaN,54.599998
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,56.400002,56.400002,2026-04-12 10:57:31.136810+00:00,NaN,54.799999
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,56.900002,56.900002,2026-04-12 10:57:31.136810+00:00,NaN,55.299999
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,57.099998,57.099998,2026-04-12 10:57:31.136810+00:00,NaN,55.500000
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,57.200001,57.200001,2026-04-12 10:57:31.136810+00:00,NaN,55.500000


## Step 1: Compute partial times between stages

Since the lifetime columns are cumulative (each includes all previous stages), we compute the time spent **between consecutive stages** by subtraction. All values in **seconds**.

| Partial column | Calculation | What it measures |
|---|---|---|
| `partial_furnace_to_2nd_strike_s` | `lifetime_2nd_strike_s` | Robot pick at furnace, transfer to main press, positioning |
| `partial_2nd_to_3rd_strike_s` | `lifetime_3rd - lifetime_2nd` | Press retraction, robot repositioning between strikes |
| `partial_3rd_to_4th_strike_s` | `lifetime_4th - lifetime_3rd` | Transfer within main press to drill station |
| `partial_4th_strike_to_auxiliary_press_s` | `lifetime_aux - lifetime_4th` | Exit main press, transfer to auxiliary press, deburring and coining |
| `partial_auxiliary_press_to_bath_s` | `lifetime_bath - lifetime_aux` | Transport from auxiliary press to quench bath |

These are the key diagnostic values: when a piece is slow, the partial that deviates identifies which segment caused the delay.

In [3]:
# TODO: implement this cell
gold = silver.copy()

gold["partial_furnace_to_2nd_s"] = gold["lifetime_2nd_strike_s"]
gold["partial_2nd_to_3rd_s"] = gold["lifetime_3rd_strike_s"] - gold["lifetime_2nd_strike_s"]
gold["partial_3rd_to_4th_s"] = gold["lifetime_4th_strike_s"] - gold["lifetime_3rd_strike_s"]
gold["partial_4th_to_aux_s"] = gold["lifetime_auxiliary_press_s"] - gold["lifetime_4th_strike_s"]
gold["partial_aux_to_bath_s"] = gold["lifetime_bath_s"] - gold["lifetime_auxiliary_press_s"]

gold.head()


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_bath_s,lifetime_general_s,processed_at,oee_cycle_time_s,lifetime_auxiliary_press_s,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,partial_aux_to_bath_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,56.200001,56.200001,2026-04-12 10:57:31.136810+00:00,NaN,54.599998,17.900000,6.700001,13.400000,16.599998,1.600002
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,56.400002,56.400002,2026-04-12 10:57:31.136810+00:00,NaN,54.799999,17.900000,6.700001,13.300001,16.899998,1.600002
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,56.900002,56.900002,2026-04-12 10:57:31.136810+00:00,NaN,55.299999,18.200001,6.599998,13.500000,17.000000,1.600002
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,57.099998,57.099998,2026-04-12 10:57:31.136810+00:00,NaN,55.500000,18.400000,6.700001,13.300001,17.099998,1.599998
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,57.200001,57.200001,2026-04-12 10:57:31.136810+00:00,NaN,55.500000,18.200001,6.599998,13.400002,17.299999,1.700001


## Step 2: Mark production gaps

Flag pieces that follow a production gap (> 5 minutes since the previous piece). Assign a `production_run_id` that groups consecutive pieces within the same uninterrupted production run.

This prevents time-series analysis from interpolating across machine stops, weekends, or maintenance periods.

In [4]:
# TODO: implement this cell
gold = gold.sort_values("timestamp").copy()

gold["gap_s"] = gold["timestamp"].diff().dt.total_seconds()
gold["production_gap"] = gold["gap_s"] > 300  # 5 minutes
gold["production_run_id"] = gold["production_gap"].cumsum()

gold[["timestamp", "gap_s", "production_gap", "production_run_id"]].head(10)


,timestamp,gap_s,production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,NaN,False,0
1,2025-11-06 15:25:29.134000+00:00,12.708,False,0
2,2025-11-06 15:25:43.743000+00:00,14.609,False,0
3,2025-11-06 15:25:56.462000+00:00,12.719,False,0
4,2025-11-06 15:26:10.462000+00:00,14.000,False,0
5,2025-11-06 15:26:23.771000+00:00,13.309,False,0
6,2025-11-06 15:26:36.382000+00:00,12.611,False,0
7,2025-11-06 15:26:50.095000+00:00,13.713,False,0
8,2025-11-06 15:27:57.427000+00:00,67.332,False,0
9,2025-11-06 15:29:04.470000+00:00,67.043,False,0


## Export to parquet

Save the gold dataset. This is the file that analytics notebooks, ML training, and the Streamlit app should consume.

In [5]:
ordered_cols = [
    "timestamp",
    "piece_id",
    "die_matrix",
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
    "partial_furnace_to_2nd_s",
    "partial_2nd_to_3rd_s",
    "partial_3rd_to_4th_s",
    "partial_4th_to_aux_s",
    "partial_aux_to_bath_s",
    "oee_cycle_time_s",
    "production_gap",
    "production_run_id",
]

gold = gold[ordered_cols]

gold.head()


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,partial_aux_to_bath_s,oee_cycle_time_s,production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,False,0
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,False,0
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,False,0
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998,NaN,False,0
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001,NaN,False,0


## Summary

In [6]:
from pathlib import Path

out_path = Path("../data/gold/pieces.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)

gold.to_parquet(out_path, index=False)
out_path


PosixPath('../data/gold/pieces.parquet')